## Description

Exploring the TACO dataset (https://github.com/pedropro/TACO/tree/master) and transforming it into the format Spare-it uses (each entry in TACO has only one segmentation, and an image that has multiple pieces of trash would have multiple corresponding entries).

Examining the TACO Dataset:

In [ ]:
import json
import requests

url = "https://raw.githubusercontent.com/pedropro/TACO/refs/heads/master/data/annotations.json"
response = requests.get(url)
data = response.json()

In [ ]:
def print_json_keys(data, indent=0, seen=None):
    if seen is None:
        seen = set()  # Initialize a set to keep track of seen structures

    # Handle dictionaries
    if isinstance(data, dict):
        keys_tuple = tuple(data.keys())  # Use a tuple of keys to identify unique structures
        if keys_tuple in seen:
            return  # Skip if this structure has already been printed
        seen.add(keys_tuple)  # Add the structure to the set

        for key in data.keys():
            print('  ' * indent + str(key))
            print_json_keys(data[key], indent + 1, seen)

    # Handle lists
    elif isinstance(data, list):
        # Detect continuous ranges
        if len(data) > 1:
            print('  ' * indent + f"[{0} - {len(data) - 1}]")  # Print the range
        else:
            for index in range(len(data)):
                print('  ' * indent + f"[{index}]")  # Print individual index if only one

        for item in data:
            print_json_keys(item, indent + 1, seen)


In [ ]:
print_json_keys(data)

info
  year
  version
  description
  contributor
  url
  date_created
images
  [0 - 1499]
    id
    width
    height
    file_name
    license
    flickr_url
    coco_url
    date_captured
    flickr_640_url
    id
    width
    height
    file_name
    license
    flickr_url
    flickr_640_url
    coco_url
    date_captured
annotations
  [0 - 4783]
    id
    image_id
    category_id
    segmentation
      [0]
        [0 - 157]
    area
    bbox
      [0 - 3]
    iscrowd
scene_annotations
  [0 - 4295]
    image_id
    background_ids
      [0]
licenses
categories
  [0 - 59]
    supercategory
    id
    name
scene_categories
  [0 - 6]
    id
    name


In [ ]:
data["scene_annotations"][0]

{'image_id': 0, 'background_ids': [1]}

In [ ]:
data["categories"]

[{'supercategory': 'Aluminium foil', 'id': 0, 'name': 'Aluminium foil'},
 {'supercategory': 'Battery', 'id': 1, 'name': 'Battery'},
 {'supercategory': 'Blister pack', 'id': 2, 'name': 'Aluminium blister pack'},
 {'supercategory': 'Blister pack', 'id': 3, 'name': 'Carded blister pack'},
 {'supercategory': 'Bottle', 'id': 4, 'name': 'Other plastic bottle'},
 {'supercategory': 'Bottle', 'id': 5, 'name': 'Clear plastic bottle'},
 {'supercategory': 'Bottle', 'id': 6, 'name': 'Glass bottle'},
 {'supercategory': 'Bottle cap', 'id': 7, 'name': 'Plastic bottle cap'},
 {'supercategory': 'Bottle cap', 'id': 8, 'name': 'Metal bottle cap'},
 {'supercategory': 'Broken glass', 'id': 9, 'name': 'Broken glass'},
 {'supercategory': 'Can', 'id': 10, 'name': 'Food Can'},
 {'supercategory': 'Can', 'id': 11, 'name': 'Aerosol'},
 {'supercategory': 'Can', 'id': 12, 'name': 'Drink can'},
 {'supercategory': 'Carton', 'id': 13, 'name': 'Toilet tube'},
 {'supercategory': 'Carton', 'id': 14, 'name': 'Other carton'

In [ ]:
data["scene_categories"][0]

{'id': 0, 'name': 'Clean'}

In [ ]:
data["info"]

{'year': 2019,
 'version': None,
 'description': 'TACO',
 'contributor': None,
 'url': None,
 'date_created': '2019-12-19T16:11:15.258399+00:00'}

In [ ]:
data["licenses"]

[]

In [40]:
data["annotations"][0]

{'id': 1,
 'image_id': 0,
 'category_id': 6,
 'segmentation': [[561.0,
   1238.0,
   568.0,
   1201.0,
   567.0,
   1175.0,
   549.0,
   1127.0,
   538.0,
   1089.0,
   519.0,
   1043.0,
   517.0,
   1005.0,
   523.0,
   964.0,
   529.0,
   945.0,
   520.0,
   896.0,
   525.0,
   862.0,
   536.0,
   821.0,
   554.0,
   769.0,
   577.0,
   727.0,
   595.0,
   678.0,
   596.0,
   585.0,
   588.0,
   346.0,
   581.0,
   328.0,
   569.0,
   306.0,
   570.0,
   276.0,
   576.0,
   224.0,
   560.0,
   205.0,
   564.0,
   170.0,
   578.0,
   154.0,
   608.0,
   136.0,
   649.0,
   127.0,
   688.0,
   127.0,
   726.0,
   129.0,
   759.0,
   141.0,
   784.0,
   153.0,
   792.0,
   177.0,
   788.0,
   193.0,
   782.0,
   209.0,
   792.0,
   238.0,
   802.0,
   271.0,
   802.0,
   294.0,
   791.0,
   319.0,
   789.0,
   360.0,
   794.0,
   395.0,
   810.0,
   529.0,
   819.0,
   609.0,
   841.0,
   675.0,
   882.0,
   728.0,
   916.0,
   781.0,
   928.0,
   802.0,
   938.0,
   834.0,
   940.0,
  

Transforming the TACO data and saving:

In [ ]:
# Same for every individual file
spare_categories = [
        {
            "supercategory": "",
            "id": 1,
            "name": "Paper Cup"
        },
        {
            "supercategory": "",
            "id": 2,
            "name": "Snack or Candy Bag or Wrapper "
        },
        {
            "supercategory": "",
            "id": 4,
            "name": "Wipe"
        },
        {
            "supercategory": "",
            "id": 5,
            "name": "Wax Paper"
        },
        {
            "supercategory": "",
            "id": 6,
            "name": "Latex Gloves"
        },
        {
            "supercategory": "",
            "id": 7,
            "name": "Juice or Other Pouch"
        },
        {
            "supercategory": "",
            "id": 8,
            "name": "Diaper"
        },
        {
            "supercategory": "",
            "id": 9,
            "name": "Padded Envelope (mixed materials)"
        },
        {
            "supercategory": "",
            "id": 10,
            "name": "Blister Pack"
        },
        {
            "supercategory": "",
            "id": 11,
            "name": "Pens and Pencils"
        },
        {
            "supercategory": "",
            "id": 14,
            "name": "Miscellaneous Office Supplies"
        },
        {
            "supercategory": "",
            "id": 15,
            "name": "Facemask and Other PPE"
        },
        {
            "supercategory": "",
            "id": 16,
            "name": "Shelf Stable Carton"
        },
        {
            "supercategory": "",
            "id": 17,
            "name": "Soiled Plastic"
        },
        {
            "supercategory": "",
            "id": 18,
            "name": "Soiled Metal"
        },
        {
            "supercategory": "",
            "id": 19,
            "name": "Soiled Glass"
        },
        {
            "supercategory": "",
            "id": 115,
            "name": "Ceramics"
        },
        {
            "supercategory": "",
            "id": 114,
            "name": "Unclassifiable"
        },
        {
            "supercategory": "",
            "id": 116,
            "name": "Filled Bag"
        },
        {
            "supercategory": "",
            "id": 117,
            "name": "Coffee Pod"
        },
        {
            "supercategory": "",
            "id": 122,
            "name": "Other Trash"
        },
        {
            "supercategory": "",
            "id": 123,
            "name": "Flexible container lid / seal"
        },
        {
            "supercategory": "",
            "id": 124,
            "name": "Snack Food Canister"
        },
        {
            "supercategory": "",
            "id": 127,
            "name": "Hard Cover Books"
        },
        {
            "supercategory": "",
            "id": 20,
            "name": "Compostable Fiber Ware"
        },
        {
            "supercategory": "",
            "id": 21,
            "name": "Compostable Cutlery"
        },
        {
            "supercategory": "",
            "id": 22,
            "name": "Compostable Plastic Cups"
        },
        {
            "supercategory": "",
            "id": 23,
            "name": "Compostable Paper Cups"
        },
        {
            "supercategory": "",
            "id": 24,
            "name": "Paper Towel/Napkins/Tissue"
        },
        {
            "supercategory": "",
            "id": 25,
            "name": "Wooden Coffee Stirrer or Chopstick"
        },
        {
            "supercategory": "",
            "id": 26,
            "name": "Soiled Cardboard Box"
        },
        {
            "supercategory": "",
            "id": 27,
            "name": "Compostable Plastic Lid"
        },
        {
            "supercategory": "",
            "id": 30,
            "name": "Food Soiled Paper"
        },
        {
            "supercategory": "",
            "id": 125,
            "name": "Other compostable material"
        },
        {
            "supercategory": "",
            "id": 126,
            "name": "Sandwich paper wrapper"
        },
        {
            "supercategory": "",
            "id": 36,
            "name": "Plastic strapping"
        },
        {
            "supercategory": "",
            "id": 38,
            "name": "Batteries"
        },
        {
            "supercategory": "",
            "id": 40,
            "name": "Cables"
        },
        {
            "supercategory": "",
            "id": 43,
            "name": "Computers"
        },
        {
            "supercategory": "",
            "id": 44,
            "name": "Monitors"
        },
        {
            "supercategory": "",
            "id": 45,
            "name": "Toner and Ink Cartridges"
        },
        {
            "supercategory": "",
            "id": 46,
            "name": "Miscellaneous Electronics "
        },
        {
            "supercategory": "",
            "id": 47,
            "name": "LED Lightbulb"
        },
        {
            "supercategory": "",
            "id": 48,
            "name": "Meat and Fish"
        },
        {
            "supercategory": "",
            "id": 49,
            "name": "Bones and Shells"
        },
        {
            "supercategory": "",
            "id": 50,
            "name": "Cheese and Other Fats"
        },
        {
            "supercategory": "",
            "id": 51,
            "name": "Fruits And Veggies"
        },
        {
            "supercategory": "",
            "id": 52,
            "name": "Other Food or Mixed Food"
        },
        {
            "supercategory": "",
            "id": 53,
            "name": "Breads"
        },
        {
            "supercategory": "",
            "id": 54,
            "name": "Grains"
        },
        {
            "supercategory": "",
            "id": 55,
            "name": "Tea Bags"
        },
        {
            "supercategory": "",
            "id": 56,
            "name": "Coffee Grounds"
        },
        {
            "supercategory": "",
            "id": 57,
            "name": "Egg Shell"
        },
        {
            "supercategory": "",
            "id": 58,
            "name": "Glass Bottles"
        },
        {
            "supercategory": "",
            "id": 59,
            "name": "Glass Jars"
        },
        {
            "supercategory": "",
            "id": 60,
            "name": "Broken Glass"
        },
        {
            "supercategory": "",
            "id": 120,
            "name": "Other Clean Glass"
        },
        {
            "supercategory": "",
            "id": 121,
            "name": "Drinking glass or glass ovenware"
        },
        {
            "supercategory": "",
            "id": 63,
            "name": "Metal Can"
        },
        {
            "supercategory": "",
            "id": 65,
            "name": "Aluminum Foil"
        },
        {
            "supercategory": "",
            "id": 66,
            "name": "Aluminum Catering Tray"
        },
        {
            "supercategory": "",
            "id": 67,
            "name": "Other Clean Metal"
        },
        {
            "supercategory": "",
            "id": 68,
            "name": "Aerosol Can"
        },
        {
            "supercategory": "",
            "id": 69,
            "name": "Metallic Bottle Cap or Lid"
        },
        {
            "supercategory": "",
            "id": 70,
            "name": "Metal Strapping"
        },
        {
            "supercategory": "",
            "id": 72,
            "name": "Liquids"
        },
        {
            "supercategory": "",
            "id": 73,
            "name": "Leaves, Flowers, Grass Clippings"
        },
        {
            "supercategory": "",
            "id": 76,
            "name": "Office Paper"
        },
        {
            "supercategory": "",
            "id": 77,
            "name": "Shredded Paper"
        },
        {
            "supercategory": "",
            "id": 78,
            "name": "Clean Cardboard"
        },
        {
            "supercategory": "",
            "id": 79,
            "name": "Refrigerated Beverage Carton"
        },
        {
            "supercategory": "",
            "id": 80,
            "name": "Magazines Newspaper"
        },
        {
            "supercategory": "",
            "id": 82,
            "name": "Receipts and Thermal Paper"
        },
        {
            "supercategory": "",
            "id": 83,
            "name": "Empty Paper Bag"
        },
        {
            "supercategory": "",
            "id": 84,
            "name": "Cardboard Coffee Cup Sleeve"
        },
        {
            "supercategory": "",
            "id": 85,
            "name": "Clean Paper Plate"
        },
        {
            "supercategory": "",
            "id": 86,
            "name": "Colored Memo Note"
        },
        {
            "supercategory": "",
            "id": 87,
            "name": "Office Folder"
        },
        {
            "supercategory": "",
            "id": 88,
            "name": "Paper Roll"
        },
        {
            "supercategory": "",
            "id": 118,
            "name": "Wrapping Paper"
        },
        {
            "supercategory": "",
            "id": 119,
            "name": "Other Clean Paper"
        },
        {
            "supercategory": "",
            "id": 89,
            "name": "Plastic Drink Bottle"
        },
        {
            "supercategory": "",
            "id": 91,
            "name": "Plastic Milk Jug or Personal Care Bottle"
        },
        {
            "supercategory": "",
            "id": 92,
            "name": "Empty Plastic Bag"
        },
        {
            "supercategory": "",
            "id": 93,
            "name": "Yogurt Tub or Container"
        },
        {
            "supercategory": "",
            "id": 94,
            "name": "Expanded Polystyrene (styrofoam)"
        },
        {
            "supercategory": "",
            "id": 95,
            "name": "Other Clean Plastics (rigid)"
        },
        {
            "supercategory": "",
            "id": 96,
            "name": "Straws"
        },
        {
            "supercategory": "",
            "id": 97,
            "name": "Clear Clamshell Container"
        },
        {
            "supercategory": "",
            "id": 98,
            "name": "Plastic Cutlery"
        },
        {
            "supercategory": "",
            "id": 99,
            "name": "Plastic Lid except black"
        },
        {
            "supercategory": "",
            "id": 100,
            "name": "Plastic Coffee Stirrer"
        },
        {
            "supercategory": "",
            "id": 101,
            "name": "Clear Plastic Cup"
        },
        {
            "supercategory": "",
            "id": 102,
            "name": "Colored Plastic Cup"
        },
        {
            "supercategory": "",
            "id": 103,
            "name": "Black Plastic"
        },
        {
            "supercategory": "",
            "id": 106,
            "name": "Plastic Wrap"
        },
        {
            "supercategory": "",
            "id": 107,
            "name": "Bubble Wrap"
        },
        {
            "supercategory": "",
            "id": 109,
            "name": "Incandescent Lightbulbs"
        },
        {
            "supercategory": "",
            "id": 110,
            "name": "CFL Lightbulbs"
        },
        {
            "supercategory": "",
            "id": 112,
            "name": "Textiles and Clothes"
        },
        {
            "supercategory": "",
            "id": 124,
            "name": "Food canister"
        }
    ]

In [ ]:
import json
import os
import datetime
from collections import defaultdict

def save_json(data, file_name, output_directory="Transformed TACO annotations"):
    """
    Saves a JSON file to the specified output directory.
    """
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    file_path = os.path.join(output_directory, file_name)
    with open(file_path, 'w') as json_file:
        json.dump(data, json_file, indent=4)


In [43]:
category_mapping = {
    31: 24,   # Tissues -> Paper Towel/Napkins/Tissue
    12: 63,   # Drink can -> Metal Can
    36: 106,  # Plastic film -> Plastic Wrap
    34: 83,   # Paper bag -> Empty Paper Bag
    25: 52,   # Food waste -> Other Food or Mixed Food
    20: 1,    # Paper cup -> Paper Cup
    1: 38,    # Battery -> Batteries
    55: 96,   # Plastic straw -> Straws
    21: 101,  # Disposable plastic cup -> Clear Plastic Cup
    30: 80,   # Magazine paper -> Magazines Newspaper
    49: 98,   # Plastic utensils -> Plastic Cutlery
    0: 65,    # Aluminium foil -> Aluminum Foil
    6: 58,    # Glass bottle -> Glass Bottles
    8: 69,    # Metal bottle cap -> Metallic Bottle Cap or Lid
    32: 118   # Wrapping paper -> Wrapping Paper
}



def map_and_transform(taco_instance):
    """
    Transforms the entire TACO JSON into individual JSON files in Spare-it format.
    Creates and saves a separate JSON file for each image.
    """
    # Current timestamp for date_created
    current_time = datetime.datetime.now().isoformat()

    # Step 1: Create a mapping from image_id to image information
    image_id_to_image = {image['id']: image for image in taco_instance['images']}

    # Step 2: Collect annotations for each image_id
    image_id_to_annotations = defaultdict(list)
    for annotation in taco_instance['annotations']:
        image_id = annotation['image_id']
        image_id_to_annotations[image_id].append(annotation)

    # Step 3: Process each image and its corresponding annotations
    for image_id, image_data in image_id_to_image.items():
        spareit_JSON = {
            "info": {
                "description": "Transformed TACO annotation",
                "url": "",
                "version": "",
                "year": 2019,
                "contributor": "",
                "date_created": current_time
            },
            "licenses": [{
                "url": "",
                "id": 0,
                "name": ""
            }],
            "images": [],
            "type": "instances",
            "annotations": [],
            "categories": spare_categories
        }

        # Transform the image data
        transformed_image = {
            'license': 0,
            'url': image_data.get('flickr_url', ""),  # where the corresponding image is hosted
            'file_name': image_data.get('file_name', "placeholder_file_name.jpg"),
            'height': image_data.get('height', 0),
            'width': image_data.get('width', 0),
            'date_captured': image_data.get('date_captured', ""),
            'id': image_id
        }
        spareit_JSON['images'].append(transformed_image)

        # Step 4: Add the corresponding annotations for this image
        annotation_id = 0
        for annotation in image_id_to_annotations[image_id]:
            # Check if annotation's category_id has a corresponding target category in the mapping
            if annotation['category_id'] in category_mapping:
                # Map to target category ID
                transformed_annotation = {
                    'id': annotation_id,
                    'image_id': 0,  # Set to 0 as per Spare-it format
                    'category_id': category_mapping[annotation['category_id']],  # Mapped category ID
                    'segmentation': annotation['segmentation'],
                    'area': annotation['area'],
                    'bbox': annotation['bbox'],
                    'iscrowd': annotation['iscrowd']
                }
                # Append the transformed annotation
                spareit_JSON['annotations'].append(transformed_annotation)
                annotation_id += 1  # Only increment if annotation is added

        # Only save the image if there are valid annotations
        if spareit_JSON['annotations']:
            # Step 5: Save the JSON file
            image_file_name = f"image_{image_id}.json"
            save_json(spareit_JSON, image_file_name)


In [44]:
map_and_transform(data)